# Extract domain models from AF2 based on DPAM-AI predictions

## Overview

This notebook converts full-length AlphaFold Database (AFDB) PDB models into a domain-centric library by splitting structures according to DPAM-AI domain predictions. Each predicted domain is extracted, capped with ACE/NME termini, renumbered to match UniProt canonical numbering, and written as a standalone PDB file with accompanying metadata.

## Workflow

1. **Read** DPAM-AI predictions (protein IDs and domain residue ranges)
2. **Truncate** full-length AF2 structures to each predicted domain range
3. **Cap** domain termini with ACE and NME groups
4. **Align** domain sequence to UniProt canonical sequence and renumber residues
5. **Write** domain PDB files and a consolidated metadata CSV

## Inputs

| Resource | Description |
|----------|-------------|
| `{subset_id}_domains` | Tab-delimited DPAM-AI output: protein IDs, domain ranges, ECOD annotations |
| `{subset_id}/{uniprot_id}.pdb` | Full-length AlphaFold 2 PDB models (one per protein) |

## Outputs

| Output | Description |
|--------|-------------|
| **Domain PDBs** | One `.pdb` file per domain. Naming: `{uniprot_id}-F1-{domain_id}_A.pdb` (e.g. `A0A024R1R8-F1-nD1_A.pdb`) |
| **`{subset_id}_domains_info.csv`** | Metadata table with columns: `id`, `uniprot_accn`, `Domain`, `Range` (UniProt numbering), `pdb_seq`, `offset`, and all DPAM-AI annotation columns |

Residue numbering in output PDBs and the `Range` column follows UniProt 1-based canonical amino acid numbering.

In [11]:
import os 
import sys
import pandas as pd
import numpy as np

wd = !git rev-parse --show-toplevel
os.chdir(wd[0])

# Directory to the DPAM-AI raw prediction (Input to this notebook)
DPAM_AI_PRED_DIR = "/scratch/diazrovi/DPAM/processing_human_afdb_v6/chunks_human_afdb"


# Directory to write the domainome .pdb and info.csv files
DOMAINOME_DIR = "./DPAM-AI_AFDB_domainome_v6"
domainome_pdb_dir = os.path.join(DOMAINOME_DIR, "pdbs")
domainome_info_dir = os.path.join(DOMAINOME_DIR, "info")

os.makedirs(domainome_pdb_dir, exist_ok=True)
os.makedirs(domainome_info_dir, exist_ok=True)

In [12]:
# ------ Helper functions ------
from Bio.PDB import PDBParser
from Bio.PDB import Structure, Model, Chain, PDBIO
from Bio.PDB import MMCIFIO
import tempfile

parser = PDBParser()

# Add python_scripts_dir to sys.path to import add_caps(u) function
python_scripts_dir = os.path.join(wd[0], "scripts", "python")
sys.path.append(python_scripts_dir)

from add_caps import add_caps

from MDAnalysis import Universe


def parse_range_to_residue_ids(range_str):
    """
    Parse Range string into set of residue IDs.
    '1-100' -> {1,2,...,100}
    '1-100,200-300' -> {1,2,...,100,200,...,300}
    """
    result = set()
    for part in str(range_str).split(","):
        part = part.strip()
        start, end = map(int, part.split("-"))
        result.update(range(start, end + 1))
    return result


def struct_to_universe(struct):
    """Convert a biopython structure to a MDAnalysis universe."""
    # Write structure to a temporary .pdb file, then load with MDAnalysis
    with tempfile.NamedTemporaryFile(suffix=".pdb", mode="w+", delete=True) as tmp:
        io = PDBIO()
        io.set_structure(struct)
        io.save(tmp.name)
        tmp.flush()
        u = Universe(tmp.name)
    return u

def universe_to_struct(u):
    """Convert an MDAnalysis universe to a biopython structure."""
    # Write universe to a temporary PDB, read back with Biopython
    with tempfile.NamedTemporaryFile(suffix=".pdb", mode="w+", delete=True) as tmp:
        u.atoms.write(tmp.name)
        tmp.flush()
        struct = parser.get_structure("converted", tmp.name)
    return struct

# Function to truncate a biopython structure to specified residue IDs (assuming single chain), add ACE and NME capping groups
def truncate_structure(struct, residue_ids):
    """
    Truncate a biopython structure to specified residue IDs (assuming single chain).
    Supports discontinuous ranges (e.g. 1-100,200-300).

    Parameters:
    - struct: Biopython Structure object
    - residue_ids: set or iterable of residue indices (1-based) to include

    Returns:
    - New Structure object containing only the specified residues
    """
    residue_ids = set(residue_ids)

    # Create a new structure to hold the truncated data
    trunc_struct = Structure.Structure(struct.id)
    # Assume only one model
    for model in struct:
        new_model = Model.Model(model.id)
        for chain in model:
            new_chain = Chain.Chain(chain.id)
            for res in chain:
                resnum = res.id[1]
                if resnum in residue_ids:
                    new_chain.add(res.copy())
            # Only add chain if it has residues
            if len(new_chain):
                new_model.add(new_chain)
        # Only add model if it has chains
        if len(new_model):
            trunc_struct.add(new_model)
        break  # Only one model

    # Add ACE and NME capping groups (add_caps handles discontinuous segments)
    u = struct_to_universe(trunc_struct)
    u = add_caps(u)
    trunc_struct = universe_to_struct(u)

    return trunc_struct

___
#### Ensure domain.pdb models are numbered according to official uniprot sequence

In [34]:
# -------- Helper functions for checking residue numberings -----------

from Bio.PDB import PDBParser


aa_name_mapping = {
    'VAL':'V', 'ILE':'I', 'LEU':'L', 'GLU':'E', 'GLN':'Q',
    'ASP':'D', 'ASN':'N', 'HIS':'H', 'TRP':'W', 'PHE':'F', 'TYR':'Y',
    'ARG':'R', 'LYS':'K', 'SER':'S', 'THR':'T', 'MET':'M', 'ALA':'A',
    'GLY':'G', 'PRO':'P', 'CYS':'C',
    'ACE': 'X', 'NME': 'X'
}

def pdb_to_sequence(structure, chain='A'):
    residues = list(structure[0][chain].get_residues())
    resnums = [residue.id[1] for residue in residues]
    assert len(resnums) == len(set(resnums)), "Residue numbers should be unique"
    seq_list = []
    if resnums:
        prev_resnum = resnums[0] - 1  # so that at the first loop there can be a gap
        for residue, resnum in zip(residues, resnums):
            gap_len = resnum - prev_resnum - 1
            if gap_len > 0:
                seq_list.extend(['-'] * gap_len)
            seq_list.append(aa_name_mapping.get(residue.resname, f"[{residue.resname}]"))
            prev_resnum = resnum
        seq = ''.join(seq_list)
    else:
        seq = ''
    return seq, resnums


def get_bfactors(structure, chain='A'):
    residues = list(structure[0][chain].get_residues())
    # bfactors = [np.mean([a.get_bfactor() for a in res.get_atoms()]) for res in residues]
    bfactors = [res['CA'].get_bfactor() for res in residues]
    return bfactors


import requests

# Function to get full UniProt entry from UniProt ID
def get_uniprot_entry(uniprot_id):
    # Input: either primaryAccession (e.g. Q9HBH1) or uniProtkbId (e.g. DEFM_HUMAN)
    url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.json"
    response = requests.get(url)
    if not response.ok:
        print("[ERROR]", response.content)
        return None
    return response.json()

# Function to get UniProt sequence from UniProt entry
def _get_uniprot_sequence(entry):
    return entry['sequence']['value'] if 'sequence' in entry else ''

# Function to get sequence from UniParc
def _get_uniparc_sequence(entry):
    uniParcId = entry['extraAttributes']['uniParcId']
    url = f"https://rest.uniprot.org/uniparc/%7Bupi%7D?upi={uniParcId}&fields=Sequence"
    response = requests.get(url)
    if not response.ok:
        print("[ERROR]", response.content)
        return None
    sequence = response.json()['sequence']['value']
    return sequence



def _get_protein_name(entry):
    try:
        if 'recommendedName' not in entry['proteinDescription']:
            assert len(entry['proteinDescription']['submissionNames']) == 1
            return entry['proteinDescription']['submissionNames'][0]['fullName']['value']
        else:
            # default case
            return entry['proteinDescription']['recommendedName']['fullName']['value']
    except (KeyError, AssertionError) as e:
        print(f"Error: Unable to fetch protein name for UniProt ID {entry['primaryAccession']}. ({e})")
        return None
    

def _get_gene_name(entry):
    try:
        return entry['genes'][0]['geneName']['value']
    except KeyError as e:
        print(f"Error: Unable to fetch gene name for UniProt ID {entry['primaryAccession']}. ({e})")
        return None


uniprot_entry = get_uniprot_entry("A0A075B6I0")
print(_get_protein_name(uniprot_entry))
print(_get_gene_name(uniprot_entry))
print(_get_uniprot_sequence(uniprot_entry))

Immunoglobulin lambda variable 8-61
IGLV8-61
MSVPTMAWMMLLLGLLAYGSGVDSQTVVTQEPSFSVSPGGTVTLTCGLSSGSVSTSYYPSWYQQTPGQAPRTLIYSTNTRSSGVPDRFSGSILGNKAALTITGAQADDESDYYCVLYMGSGI


___
#### Ensure the domain .pdb model always follow the UniProt fasta sequence's residue number

In [ ]:
# TODO: Worklow to extract pdb_seq and resnums, compare with uniprot_seq, compute the residue number offset 
# between pdb_seq and uniprot_seq, and apply the offset to create a copy of the struct with residues renumbered 
# according to the uniprot sequence

In [29]:
# Function to run EMBOSS Needle alignment
import subprocess
import tempfile

def neelde_alignment(seq1, seq2):
    """
    Run EMBOSS needle alignment between two sequences (seq1, seq2).
    Returns the full stdout containing the alignment.
    """
    with tempfile.NamedTemporaryFile(mode="w", delete=False) as f1, tempfile.NamedTemporaryFile(mode="w", delete=False) as f2, tempfile.NamedTemporaryFile(mode="r") as outf:
        # Write sequences to temp files in fasta format
        f1.write(f">seq1\n{seq1}\n")
        f1.flush()
        f2.write(f">seq2\n{seq2}\n")
        f2.flush()

        # Prepare the command. Using '-auto' for non-interactive, '-aformat3=pair' for readable output
        # Output is written to outf.name
        cmd = [
            "needle",
            "-asequence", f1.name,
            "-bsequence", f2.name,
            "-sprotein1",
            "-sprotein2",
            "-gapopen", "10",
            "-gapextend", "0.5",
            "-outfile", outf.name,
            "-auto"
        ]
        try:
            subprocess.run(cmd, check=True, capture_output=True)
            alignment = outf.read()
        except subprocess.CalledProcessError as e:
            print(f"Error running EMBOSS needle: {e.stderr.decode() if e.stderr else e}")
            return None

    return alignment


# Function to align pdb_seq to uniprot_seq, and return a mapping of pdb_resnums to uniprot_resnums (1-based)
def map_pdb_resnums_to_uniprot(pdb_seq, pdb_resnums, uniprot_seq):
    """
    Aligns a PDB sequence to a UniProt sequence using global alignment (EMBOSS needle) and returns a mapping between residue numbers in the PDB structure and their corresponding residue numbers in the UniProt sequence.

    Parameters
    ----------
    pdb_seq : str
        The sequence extracted from the PDB structure (e.g., derived from coordinates), containing only the residues present in the PDB.
    pdb_resnums : List[int]
        The list of residue numbers from the PDB structure, corresponding to each amino acid in pdb_seq. PDB residue numbering can include insertion codes or non-contiguous values.
    uniprot_seq : str
        The canonical full-length sequence from UniProt for the same protein.

    Returns
    -------
    mapping : List[Tuple[int, int]]
        A list of (pdb_resnum, uniprot_resnum) pairs, where each tuple indicates that pdb_resnum (from pdb_resnums) is aligned to uniprot_resnum in the UniProt sequence (both 1-based).
        - Residues corresponding to gaps in either sequence are not included.
        - Non-standard PDB residues ('X' codes) are skipped and not mapped.
        - Mappings rely on the global alignment and do not necessarily assume perfect identity.

    Notes
    -----
    - The function will skip positions where: the PDB residue is a gap ('-'), maps to a UniProt gap ('-'), or is an 'X' residue.
    - The mapping allows you to trace which residue in the PDB structure corresponds to which residue in the reference UniProt sequence, accounting for possible missing residues or sequence discrepancies.
    - Useful for translating PDB-based features (e.g., 3D coordinates, post-translational modifications, predictions) back to UniProt numbering.

    Example
    -------
    mapping = map_pdb_resnums_to_uniprot(
        pdb_seq="MQQ...A",
        pdb_resnums=[10,11,...,40],
        uniprot_seq="MQQS...AA"
    )
    # mapping will look like: [(10, 15), (11, 16), ..., (40, 37)]
    """
    from Bio import AlignIO
    import io

    alignment = neelde_alignment(pdb_seq, uniprot_seq)
    if alignment is None:
        return []

    aln = AlignIO.read(io.StringIO(alignment), "emboss")
    seq1_aligned = str(aln[0].seq)  # PDB (seq1 in needle)
    seq2_aligned = str(aln[1].seq)  # UniProt (seq2 in needle)

    mapping = []
    pdb_idx = 0
    uniprot_resnum = 0

    for i in range(len(seq1_aligned)):
        c1 = seq1_aligned[i]
        c2 = seq2_aligned[i]

        if c1 == "-":
            if c2 != "-":
                uniprot_resnum += 1
            continue
        if c2 == "-":
            pdb_idx += 1
            continue

        # Both have letters
        uniprot_resnum += 1
        if c1 == "X":
            pdb_idx += 1
            continue

        mapping.append((pdb_resnums[pdb_idx], uniprot_resnum))
        pdb_idx += 1

    return mapping


# Validation: test with current pdb_seq, pdb_resnums, uniprot_seq (requires cells 7 and 12 to be run first)
# mapping = map_pdb_resnums_to_uniprot(pdb_seq, pdb_resnums, uniprot_seq)
# print(mapping)


# Compute offset based on mapping - use the most common offset among all residue pairs between pdb and uniprot
import statistics

def mode_offset(mapping):
    """
    Compute the mode of the offset between pdb_resnums and uniprot_resnums.
    """
    offsets = [u - p for p, u in mapping]
    return statistics.mode(offsets)

# mode_offset(mapping)


# Function to create a copy of the structure with renumbered residues
def renumber_structure(struct, offset):
    """
    Create a copy of the structure with residues renumbered according to the offset.

    None is treated as 0.
    """
    if offset is None:
        offset = 0
    new_struct = struct.copy()
    for model in new_struct:
        for chain in model:
            for res in chain:
                res.id = (res.id[0], res.id[1] + offset, res.id[2])
    return new_struct


def _max_residue_number(struct):
    """Return the maximum residue number in the structure (any chain)."""
    max_resid = 0
    for model in struct:
        for chain in model:
            for res in chain:
                max_resid = max(max_resid, res.id[1])
    return max_resid

In [15]:
# ---- Proof-of-concept workflow ----

# Example renumbering workflow
struct = parser.get_structure("converted", "/scratch/ymeng/DPAM/DPAM-AI_AFDB_domainome_v6/pdbs/A0A024RBG1-F1-nD1_A.pdb")

# # Test: apply offset to struct to see if it can detect it
# struct = renumber_structure(struct, 5)

# Get sequence of the pdb structure
pdb_seq, pdb_resnums = pdb_to_sequence(struct)


# Get canonical uniprot sequence
uniprot_entry = get_uniprot_entry("A0A024RBG1")
uniprot_seq = _get_uniprot_sequence(uniprot_entry)

# Perform alignment and map pdb_resnums to uniprot_resnums
# Compute offset based on mapping - use the most common offset among all residue pairs between pdb and uniprot
alignment = neelde_alignment(pdb_seq, uniprot_seq)
mapping = map_pdb_resnums_to_uniprot(pdb_seq, pdb_resnums, uniprot_seq)
offset = mode_offset(mapping)

print(mapping)
print(offset)

# Apply the offset
struct_renumbered = renumber_structure(struct, offset)
pdb_seq, pdb_resnums = pdb_to_sequence(struct_renumbered)
print(pdb_seq)
print(pdb_resnums)


[(11, 11), (12, 12), (13, 13), (14, 14), (15, 15), (16, 16), (17, 17), (18, 18), (19, 19), (20, 20), (21, 21), (22, 22), (23, 23), (24, 24), (25, 25), (26, 26), (27, 27), (28, 28), (29, 29), (30, 30), (31, 31), (32, 32), (33, 33), (34, 34), (35, 35), (36, 36), (37, 37), (38, 38), (39, 39), (40, 40), (41, 41), (42, 42), (43, 43), (44, 44), (45, 45), (46, 46), (47, 47), (48, 48), (49, 49), (50, 50), (51, 51), (52, 52), (53, 53), (54, 54), (55, 55), (56, 56), (57, 57), (58, 58), (59, 59), (60, 60), (61, 61), (62, 62), (63, 63), (64, 64), (65, 65), (66, 66), (67, 67), (68, 68), (69, 69), (70, 70), (71, 71), (72, 72), (73, 73), (74, 74), (75, 75), (76, 76), (77, 77), (78, 78), (79, 79), (80, 80), (81, 81), (82, 82), (83, 83), (84, 84), (85, 85), (86, 86), (87, 87), (88, 88), (89, 89), (90, 90), (91, 91), (92, 92), (93, 93), (94, 94), (95, 95), (96, 96), (97, 97), (98, 98), (99, 99), (100, 100), (101, 101), (102, 102), (103, 103), (104, 104), (105, 105), (106, 106), (107, 107), (108, 108), (

___
### Example workflow:

___
### Master function to process a whole subset

In [36]:
from Bio.PDB import PDBIO

# Master function to process one subset
def subset_to_domainome(subset_id, domainome_pdb_dir, parser):
    """
    Process one subset to:
    - Read DPAM-AI output for specified subset_id.
    - For each unique uniprot_accn, write truncated PDB domain files.
    - Stream domain info to a domainome_info.csv file.
    """

    # Paths for this subset
    job_dir = os.path.join(DPAM_AI_PRED_DIR, f"job_{subset_id}")
    job_input_dir = os.path.join(job_dir, subset_id)
    job_pred_path = os.path.join(job_dir, f"{subset_id}_domains")
    domainome_info_path = os.path.join(DOMAINOME_DIR, "info", f"{subset_id}_domains_info.csv")

    # Read the DPAM-AI output
    df_pred = pd.read_csv(job_pred_path, sep="\t")

    # Rename `Protein` column to `uniprot_accn`
    df_pred.rename(columns={"Protein": "uniprot_accn"}, inplace=True)

    # List unique uniprot_accn
    unique_uniprot = df_pred["uniprot_accn"].unique()

    # Add `id` column and move it to the first column
    df_pred["id"] = df_pred["uniprot_accn"] + "-F1-" + df_pred["Domain"].astype(str) + "_A"
    df_pred.insert(0, "id", df_pred.pop("id"))

    # Initialize the header for domainome_info.csv
    header = df_pred.head(0)

    # Initialize the additional columns to header
    header["offset"] = None
    header["pdb_seq"] = None
    header.to_csv(domainome_info_path, index=False)

    # For each unique uniprot_accn
    for uniprot_accn in unique_uniprot:
        # Subset the DPAM-AI output for this uniprot_accn
        df_pred_uniprot = df_pred[df_pred["uniprot_accn"] == uniprot_accn]

        # Read the FL AF2 .pdb model for this uniprot_accn
        AF2_pdb_path = os.path.join(job_input_dir, f"{uniprot_accn}.pdb")
        AF2_struct = parser.get_structure(uniprot_accn, AF2_pdb_path)

        # Fetch UniProt sequence once per protein (used for all domains)
        uniprot_entry = get_uniprot_entry(uniprot_accn)
        uniprot_seq = _get_uniprot_sequence(uniprot_entry) if uniprot_entry else ""
        if not uniprot_seq:
            try: 
                # Fallback to UniParc if UniProt fails
                uniprot_seq = _get_uniparc_sequence(uniprot_entry)
            except Exception as e:
                raise ValueError(f"Could not fetch UniProt sequence for {uniprot_accn}: {e}")

        # For each row, write the domain .pdb file, and stream the domain info to domainome_info.csv
        for _, row in df_pred_uniprot.iterrows():
            domain_id = row["id"]
            domain_range = row["Range"]

            # Parse Range (supports "1-100" or "1-100,200-300")
            residue_ids = parse_range_to_residue_ids(domain_range)
            domain_struct = truncate_structure(AF2_struct, residue_ids)

            # ----- Ensure residue numbers match UniProt 1-based numbering -----
            pdb_seq, pdb_resnums = pdb_to_sequence(domain_struct)
            row["pdb_seq"] = pdb_seq

            mapping = map_pdb_resnums_to_uniprot(pdb_seq, pdb_resnums, uniprot_seq)
            offset = mode_offset(mapping) if mapping else None

            domain_struct_renumbered = renumber_structure(domain_struct, offset)
            max_resid = _max_residue_number(domain_struct_renumbered)

            # Handle edge case where the residue number after alignment is too large to save as .pdb format
            if max_resid > 9999:
                struct_to_save = domain_struct
                row["offset"] = "NA"
                row["Range"] = domain_range
            else:
                struct_to_save = domain_struct_renumbered
                effective_offset = offset if offset is not None else 0
                ranges = []
                for part in str(domain_range).split(","):
                    start, end = map(int, part.strip().split("-"))
                    ranges.append((start + effective_offset, end + effective_offset))
                row["Range"] = ",".join(f"{s}-{e}" for s, e in ranges)
                row["offset"] = offset

            # Write the domain .pdb file
            domain_pdb_path = os.path.join(domainome_pdb_dir, f"{domain_id}.pdb")
            io = PDBIO()
            io.set_structure(struct_to_save)
            io.save(domain_pdb_path)

            # Stream the row to domainome_info.csv (enforce column order to match header)
            with open(domainome_info_path, "a") as f:
                row_df = row.to_frame().T
                row_df = row_df[header.columns]
                row_df.to_csv(f, header=False, index=False)


# Example usage
subset_id = "0117"
subset_to_domainome(subset_id, domainome_pdb_dir, parser)

___
### SLURM: process the entire database

Workflow of this notebook is refactored to:
- `scripts/python/DPAM_domains_to_pdb.py` CLI entry point
- `scripts/slurm/DPAM_doamins_to_pdb.sh` SLURM submission script

Submit to SLURM to process the entire database

In [37]:
!sbatch scripts/slurm/DPAM_domains_to_pdb.sh

sbatch: [ESTIMATION] The estimated cost of this job is CHF 0.00
sbatch: ╭──────────────────────────────┬─────────────┬─────────────┬─────────────╮
sbatch: │ [in CHF]                     │ Capping     │ Consumed    │ Queued ¹⁾ ²⁾│
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ username : ymeng             │ 0           │ 47.5        │ 0.05        │
sbatch: ├──────────────────────────────┼─────────────┼─────────────┼─────────────┤
sbatch: │ account : upthomae           │ 10,000      │ 93.3        │ 0.05        │
sbatch: ╰──────────────────────────────┴─────────────┴─────────────┴─────────────╯
sbatch: ¹⁾ Estimated cost of the queued jobs and this job
sbatch: ²⁾ Queued jobs costs are based on its walltime (option --time)
Submitted batch job 50122283


In [17]:
# Check for any error in the .out files
!grep -l -E "ERROR|Traceback|Error:|Exception|failed|ValueError|KeyError" /scratch/ymeng/DPAM/logs/DPAM_domains_to_pdb_50119995/domains_to_pdb-50119995_*.out

/scratch/ymeng/DPAM/logs/DPAM_domains_to_pdb_50119995/domains_to_pdb-50119995_117.out
/scratch/ymeng/DPAM/logs/DPAM_domains_to_pdb_50119995/domains_to_pdb-50119995_168.out
/scratch/ymeng/DPAM/logs/DPAM_domains_to_pdb_50119995/domains_to_pdb-50119995_515.out
/scratch/ymeng/DPAM/logs/DPAM_domains_to_pdb_50119995/domains_to_pdb-50119995_846.out
/scratch/ymeng/DPAM/logs/DPAM_domains_to_pdb_50119995/domains_to_pdb-50119995_960.out
/scratch/ymeng/DPAM/logs/DPAM_domains_to_pdb_50119995/domains_to_pdb-50119995_961.out


In [40]:
# Check the number of files are as expected
!ls /scratch/ymeng/DPAM/logs/DPAM_domains_to_pdb_50119995 | wc -l
!ls /scratch/ymeng/DPAM/DPAM-AI_AFDB_domainome_v6/info | wc -l
!ls /scratch/ymeng/DPAM/DPAM-AI_AFDB_domainome_v6/pdbs | wc -l

1000
1000
52513


In [39]:
# Combine all the info.csv files into one
domain_info_paths = os.listdir(domainome_info_dir)
df_domain_info_all = pd.DataFrame()

for path in domain_info_paths:
    df_domain_info = pd.read_csv(os.path.join(domainome_info_dir, path))
    df_domain_info_all = pd.concat([df_domain_info_all, df_domain_info])

df_domain_info_all.to_csv(os.path.join(DOMAINOME_DIR, "DPAM_AI_AFDB_domainome_v6_domains_info.csv"), index=False)

# Check the shape of this dataframe
print(f"df_domain_info_all.shape: {df_domain_info_all.shape}")

df_domain_info_all.shape: (52513, 17)


In [41]:
df_domain_info_all.iloc[0]

id                                                Q8WVQ1-F1-nD1_A
uniprot_accn                                               Q8WVQ1
Domain                                                        nD1
Range                                                      86-400
ECOD_num                                                     2167
ECOD_key                                                  e1s1dA1
T-group                                                     5.1.2
DPAM_prob                                                   0.979
HH_prob                                                       1.0
DALI_zscore                                                  57.6
Hit_cov                                                     0.994
Tgroup_cov                                                  0.969
Judge                                                 good_domain
Hcount                                                          3
Scount                                                         25
offset    

In [53]:
# ------ Double check that the offset is applied correctly -----

# Check if there are any rows for which offset is not 0
df_offset = df_domain_info_all[df_domain_info_all["offset"] != 0]

row = df_offset.iloc[5]
print(row)

domain_id = row["id"]
uniprot_accn = row["uniprot_accn"]
domain_range = row["Range"]
domain_seq = row["pdb_seq"]

# Get the uniprot sequence
uniprot_entry = get_uniprot_entry(uniprot_accn)
uniprot_seq = _get_uniprot_sequence(uniprot_entry)

# parse_range_to_residue_ids returns a set; need sorted list for indexing
residue_ids = sorted(parse_range_to_residue_ids(domain_range))
print(f"residue_ids: {residue_ids}")

# Use sorted residue_ids to get contiguous segment from UniProt sequence
start_res = residue_ids[0]
end_res = residue_ids[-1]
domain_seq_uniprot = uniprot_seq[start_res - 1:end_res]  # Python 0-based

# Remove 'X' from domain_seq
domain_seq_no_X = domain_seq.replace('X', '')

# Compare domain_seq and domain_seq_uniprot
print(f"domain_seq:         {domain_seq_no_X}")
print(f"domain_seq_uniprot: {domain_seq_uniprot}")



id                                               P17927-F1-nD15_A
uniprot_accn                                               P17927
Domain                                                       nD15
Range                                                     491-550
ECOD_num                                                  1811351
ECOD_key                                                  e5fobC3
T-group                                                   389.1.2
DPAM_prob                                                     1.0
HH_prob                                                     0.998
DALI_zscore                                                   7.2
Hit_cov                                                     0.877
Tgroup_cov                                                  0.952
Judge                                                 good_domain
Hcount                                                          0
Scount                                                          4
offset    